# Detecção de fraude em cartão de crédito

Projeto do desafio da DIO: treinar modelos que detectam fraude em transações reais de cartão de crédito, onde **só 0,17% das transações são fraude** e, por isso, a acurácia engana.

**Roteiro**

1. Explorar a base e medir o desbalanceamento
2. Preparar os dados (log do valor, padronização, divisão estratificada)
3. Treinar e comparar: regressão logística (baseline), Random Forest e XGBoost
4. Avaliar pelas métricas da classe de fraude: recall, precisão, F1, curva ROC e curva de precisão e recall
5. Ajustar: Pipeline, limiar de decisão e GridSearchCV
6. Evoluir: undersampling × oversampling, Random Forest no Pipeline com ajuste de limiar, busca maior de hiperparâmetros, variáveis de tempo e outros modelos
7. Explicar: importância das variáveis e SHAP

A base é lida direto do link do desafio, então o notebook roda no Google Colab sem baixar nada.

In [ ]:
# XGBoost e SHAP já vêm no Colab; a linha abaixo instala só se faltar.
import importlib.util, subprocess, sys
for pacote in ["xgboost", "shap"]:
    if importlib.util.find_spec(pacote) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pacote])

import warnings
warnings.filterwarnings("ignore")

from time import perf_counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, fbeta_score,
                             roc_auc_score, average_precision_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay, roc_curve, precision_recall_curve)
from xgboost import XGBClassifier
import shap

SEED = 42
np.random.seed(SEED)
sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
pd.set_option("display.max_columns", 40)

## 1. Explorar

`Time` é o número de segundos desde a primeira transação da base (cobre dois dias), `Amount` é o valor e `Class` é a resposta (1 = fraude, 0 = normal). `V1` a `V28` vieram de um PCA aplicado pelo banco para proteger a privacidade, então não têm nome de negócio.

In [ ]:
DATA_URL = "https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv"
df = pd.read_csv(DATA_URL)
print("Formato:", df.shape)
df.head()

In [ ]:
print("Valores ausentes:", int(df.isnull().sum().sum()))
print("Linhas duplicadas:", int(df.duplicated().sum()))
print("Fraudes entre as duplicadas:", int(df[df.duplicated()]["Class"].sum()))
df[["Time", "Amount", "Class"]].describe()

In [ ]:
proporcao = pd.DataFrame({
    "transações": df["Class"].value_counts(),
    "proporção": df["Class"].value_counts(normalize=True),
}).rename(index={0: "normal (0)", 1: "fraude (1)"})
proporcao

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.2))

contagem = df["Class"].value_counts().sort_index()
axes[0].bar(["normal", "fraude"], contagem.values, color=["#4C72B0", "#C44E52"])
axes[0].set_yscale("log")
axes[0].set_title("Transações por classe (escala log)")
for i, v in enumerate(contagem.values):
    axes[0].text(i, v * 1.15, f"{v:,}".replace(",", "."), ha="center")

for classe, cor, rotulo in [(0, "#4C72B0", "normal"), (1, "#C44E52", "fraude")]:
    axes[1].hist(np.log1p(df.loc[df["Class"] == classe, "Amount"]), bins=50, density=True,
                 alpha=0.6, color=cor, label=rotulo)
axes[1].set_title("Distribuição de log(1 + Amount)")
axes[1].set_xlabel("log(1 + valor)")
axes[1].legend()

hora = (df["Time"] // 3600) % 24
taxa_hora = df.groupby(hora)["Class"].mean() * 100
axes[2].bar(taxa_hora.index, taxa_hora.values, color="#C44E52")
axes[2].set_title("Taxa de fraude por hora relativa (%)")
axes[2].set_xlabel("hora desde o início da coleta (mod 24)")
plt.tight_layout()
plt.show()

**O que a exploração mostra**

- Não há valores ausentes, mas há linhas duplicadas. Removo as duplicadas para que a mesma transação não caia no treino e no teste ao mesmo tempo, o que inflaria as métricas.
- A fraude é rara (cerca de 0,17%). Um modelo que responde "normal" para tudo já acerta mais de 99,8%.
- A taxa de fraude muda ao longo do dia: nas horas de madrugada (pouco movimento), a proporção de fraude sobe bastante. Isso motiva as variáveis de tempo da seção 6.

## 2. Preparar

- `Amount_log = log1p(Amount)`: o valor é muito assimétrico (muitas compras pequenas e poucas enormes); o log comprime a cauda.
- Variáveis de tempo (usadas só na seção 6.4): hora relativa, segundos desde a transação anterior, número de transações no último minuto e valor comparado com a média das 10 transações anteriores. A base não tem identificador de cartão, então essas variáveis descrevem o movimento geral do sistema, não o de um cliente. Todas usam só o passado de cada transação.
- Divisão treino/teste 80/20 com `stratify=y`, para manter a proporção de fraudes nos dois.
- `StandardScaler` ajustado **só no treino** e aplicado no teste.

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)
print("Formato sem duplicadas:", df.shape, "| fraudes:", int(df["Class"].sum()))

df["Amount_log"] = np.log1p(df["Amount"])

# Variáveis de comportamento ao longo do tempo (só olham para o passado)
ordem = df.sort_values("Time", kind="mergesort")
tempo = pd.to_datetime(ordem["Time"], unit="s")
serie_um = pd.Series(1.0, index=tempo)
df.loc[ordem.index, "Hour"] = ((ordem["Time"] // 3600) % 24).values
df.loc[ordem.index, "secs_since_prev"] = ordem["Time"].diff().fillna(0).values
df.loc[ordem.index, "tx_last_60s"] = serie_um.rolling("60s").sum().values
media_10 = ordem["Amount_log"].shift(1).rolling(10, min_periods=1).mean()
df.loc[ordem.index, "amount_vs_last10"] = (ordem["Amount_log"] - media_10).fillna(0).values

V_COLS = [f"V{i}" for i in range(1, 29)]
FEATURES = V_COLS + ["Amount_log"]
TIME_FEATURES = ["Hour", "secs_since_prev", "tx_last_60s", "amount_vs_last10"]

train_df, test_df = train_test_split(df, test_size=0.2, stratify=df["Class"], random_state=SEED)
X_train, X_test = train_df[FEATURES], test_df[FEATURES]
y_train, y_test = train_df["Class"].values, test_df["Class"].values

scaler = StandardScaler().fit(X_train)          # ajusta só no treino
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

pd.DataFrame({
    "linhas": [len(y_train), len(y_test)],
    "fraudes": [y_train.sum(), y_test.sum()],
    "proporção de fraude": [y_train.mean(), y_test.mean()],
}, index=["treino", "teste"])

## 3. Treinar

Antes dos modelos de verdade, um classificador "burro" que sempre responde **normal** mostra a armadilha da acurácia.

Uma função de avaliação calcula as métricas da **classe de fraude**:

- **recall**: das fraudes reais, quantas o modelo encontrou;
- **precisão**: das transações marcadas como fraude, quantas eram fraude;
- **F1**: média harmônica das duas;
- **PR-AUC (average precision)**: área sob a curva de precisão e recall, que resume o modelo em todos os limiares e não se deixa inflar pela classe majoritária como a ROC-AUC.

Os modelos de árvore recebem os dados sem padronizar (árvores não dependem de escala e, assim, o SHAP mostra os valores originais). A regressão logística recebe os dados padronizados.

In [ ]:
def avaliar(nome, y_true, proba, limiar=0.5):
    pred = (proba >= limiar).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return {
        "modelo": nome,
        "limiar": limiar,
        "acurácia": accuracy_score(y_true, pred),
        "precisão": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "F1": f1_score(y_true, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, proba),
        "PR-AUC": average_precision_score(y_true, proba),
        "fraudes pegas": f"{tp}/{tp + fn}",
        "falsos positivos": int(fp),
    }

dummy = DummyClassifier(strategy="most_frequent").fit(X_train_s, y_train)
pd.DataFrame([avaliar("Sempre 'normal'", y_test, dummy.predict_proba(X_test_s)[:, 1])])

Acurácia de 99,8% e **zero fraudes encontradas**. É por isso que a partir daqui o foco é recall, precisão e F1 da classe 1.

In [ ]:
neg, pos = np.bincount(y_train)
peso_xgb = neg / pos
print(f"Normais por fraude no treino: {peso_xgb:.0f}")

modelos = {
    "Regressão logística": (LogisticRegression(max_iter=1000, random_state=SEED), "escalado"),
    "Regressão logística (balanced)": (LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED), "escalado"),
    "Random Forest (balanced)": (RandomForestClassifier(n_estimators=200, class_weight="balanced", min_samples_leaf=2,
                                                        n_jobs=-1, random_state=SEED), "bruto"),
    "XGBoost": (XGBClassifier(n_estimators=400, max_depth=5, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                              eval_metric="aucpr", tree_method="hist", n_jobs=-1, random_state=SEED), "bruto"),
    "XGBoost (scale_pos_weight)": (XGBClassifier(n_estimators=400, max_depth=5, learning_rate=0.05, subsample=0.8,
                                                 colsample_bytree=0.8, scale_pos_weight=peso_xgb, eval_metric="aucpr",
                                                 tree_method="hist", n_jobs=-1, random_state=SEED), "bruto"),
}

probas, tempos = {}, {}
for nome, (modelo, entrada) in modelos.items():
    Xtr, Xte = (X_train_s, X_test_s) if entrada == "escalado" else (X_train, X_test)
    inicio = perf_counter()
    modelo.fit(Xtr, y_train)
    tempos[nome] = perf_counter() - inicio
    probas[nome] = modelo.predict_proba(Xte)[:, 1]
    print(f"{nome:<32} treinado em {tempos[nome]:6.1f}s")

resultados = pd.DataFrame([avaliar(n, y_test, p) for n, p in probas.items()])
resultados

## 4. Avaliar

Relatório de classificação, matrizes de confusão, curva ROC e curva de precisão e recall.

In [ ]:
for nome, proba in probas.items():
    print(f"=== {nome} (limiar 0,5) ===")
    print(classification_report(y_test, (proba >= 0.5).astype(int), target_names=["normal", "fraude"], digits=4))

In [ ]:
fig, axes = plt.subplots(1, len(probas), figsize=(4.2 * len(probas), 3.8))
for ax, (nome, proba) in zip(axes, probas.items()):
    ConfusionMatrixDisplay(confusion_matrix(y_test, (proba >= 0.5).astype(int)),
                           display_labels=["normal", "fraude"]).plot(ax=ax, colorbar=False, values_format="d")
    ax.set_title(nome, fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
fig, (ax_roc, ax_pr) = plt.subplots(1, 2, figsize=(15, 5.5))
for nome, proba in probas.items():
    fpr, tpr, _ = roc_curve(y_test, proba)
    ax_roc.plot(fpr, tpr, label=f"{nome} (AUC={roc_auc_score(y_test, proba):.3f})")
    prec, rec, _ = precision_recall_curve(y_test, proba)
    ax_pr.plot(rec, prec, label=f"{nome} (AP={average_precision_score(y_test, proba):.3f})")
ax_roc.plot([0, 1], [0, 1], "k--", lw=1)
ax_roc.set(title="Curva ROC", xlabel="taxa de falsos positivos", ylabel="recall (taxa de verdadeiros positivos)")
ax_pr.axhline(y_test.mean(), color="k", ls="--", lw=1, label=f"aleatório ({y_test.mean():.4f})")
ax_pr.set(title="Curva de precisão e recall", xlabel="recall", ylabel="precisão")
ax_roc.legend(fontsize=8, loc="lower right"); ax_pr.legend(fontsize=8, loc="lower left")
plt.tight_layout()
plt.show()

**Como ler**

- Na **curva ROC** todos os modelos parecem excelentes (AUC acima de 0,95). Isso acontece porque o eixo x é a taxa de falsos positivos sobre ~57 mil transações normais: centenas de alarmes falsos ainda dão uma taxa minúscula.
- A **curva de precisão e recall** é a que separa os modelos: a linha tracejada (sorteio) fica em 0,17%, e quanto mais a curva se mantém alta à direita, mais fraudes o modelo pega sem encher a fila de alarmes falsos.
- Dar peso à fraude (`class_weight="balanced"` na regressão logística) sobe o recall, mas derruba a precisão: o modelo passa a marcar milhares de transações normais.

## 5. Ajustar

### 5.1 Pipeline + limiar de decisão

O `Pipeline` junta padronização e modelo, então o `StandardScaler` é reajustado dentro de cada dobra da validação cruzada e nunca vê dados de teste. Depois, comparo o limiar padrão de 0,5 com o limiar de 0,3 usado pela Expert.

In [ ]:
pipe_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("modelo", LogisticRegression(max_iter=1000, random_state=SEED)),
])
pipe_lr.fit(X_train, y_train)
proba_pipe_lr = pipe_lr.predict_proba(X_test)[:, 1]
pd.DataFrame([avaliar("LR Pipeline", y_test, proba_pipe_lr, 0.5),
              avaliar("LR Pipeline", y_test, proba_pipe_lr, 0.3)])

In [ ]:
def curva_limiar(y_true, proba, titulo, ax, marcar=(0.3, 0.5)):
    limiares = np.linspace(0.01, 0.99, 99)
    linhas = [avaliar("", y_true, proba, t) for t in limiares]
    ax.plot(limiares, [l["precisão"] for l in linhas], label="precisão")
    ax.plot(limiares, [l["recall"] for l in linhas], label="recall")
    ax.plot(limiares, [l["F1"] for l in linhas], label="F1")
    for t in marcar:
        ax.axvline(t, color="gray", ls=":", lw=1)
    ax.set(title=titulo, xlabel="limiar de decisão", ylim=(0, 1.02))
    ax.legend()

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
curva_limiar(y_test, proba_pipe_lr, "Regressão logística: métricas × limiar", axes[0])
curva_limiar(y_test, probas["XGBoost"], "XGBoost: métricas × limiar", axes[1])
plt.tight_layout()
plt.show()

Baixar o limiar troca precisão por recall: mais fraudes entram na rede, junto com mais alarmes falsos. O quanto vale a pena depende do custo de cada erro (uma fraude perdida custa o valor da compra; um alarme falso custa uma ligação ou um cliente irritado).

### 5.2 GridSearchCV escolhendo pelo recall

A busca usa validação cruzada estratificada de 5 dobras dentro do treino. Peço recall como critério de escolha (`refit="recall"`), como no desafio, mas registro também precisão, F1 e PR-AUC de cada combinação para ver o preço dessa escolha. A grade é maior que a da aula: 5 valores de `C` × 4 opções de peso para a fraude.

In [ ]:
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
grade_lr = {
    "modelo__C": [0.001, 0.01, 0.1, 1, 10],
    "modelo__class_weight": [None, {0: 1, 1: 10}, {0: 1, 1: 50}, "balanced"],
}
grid_lr = GridSearchCV(pipe_lr, grade_lr, cv=cv5, n_jobs=-1, refit="recall",
                       scoring={"recall": "recall", "precisão": "precision", "F1": "f1", "PR-AUC": "average_precision"})
inicio = perf_counter()
grid_lr.fit(X_train, y_train)
print(f"GridSearchCV em {perf_counter() - inicio:.0f}s | melhor por recall: {grid_lr.best_params_}")

cv_res = pd.DataFrame(grid_lr.cv_results_)
tabela_grid = pd.DataFrame({
    "C": cv_res["param_modelo__C"],
    "class_weight": cv_res["param_modelo__class_weight"].astype(str),
    "recall": cv_res["mean_test_recall"],
    "precisão": cv_res["mean_test_precisão"],
    "F1": cv_res["mean_test_F1"],
    "PR-AUC": cv_res["mean_test_PR-AUC"],
}).sort_values("recall", ascending=False)
tabela_grid.head(10)

In [ ]:
melhor_f1 = tabela_grid.sort_values("F1", ascending=False).iloc[0]
print("Melhor combinação por F1 na validação cruzada:")
print(melhor_f1.to_string())

proba_grid_lr = grid_lr.best_estimator_.predict_proba(X_test)[:, 1]
pd.DataFrame([avaliar("LR GridSearch (recall)", y_test, proba_grid_lr, 0.5)])

Escolher **só** pelo recall leva a grade para o peso mais alto da fraude e para regularização forte: o recall sobe, mas a precisão cai para poucos por cento, ou seja, a maioria dos alertas é alarme falso. Por isso, na escolha final (seção 6.2) uso o recall como prioridade, mas com uma métrica que também cobra precisão.

## 6. Ideias para evoluir

### 6.1 Undersampling × oversampling

A amostragem é feita **só no treino**; o teste continua com a proporção real de fraude.

- **Undersampling**: mantém todas as fraudes e sorteia o mesmo número de normais (~750 linhas no total).
- **Oversampling**: mantém todas as normais e repete fraudes sorteadas com reposição até empatar (~450 mil linhas).

In [ ]:
rng = np.random.default_rng(SEED)
idx_fraude = np.where(y_train == 1)[0]
idx_normal = np.where(y_train == 0)[0]
idx_under = np.concatenate([idx_fraude, rng.choice(idx_normal, size=len(idx_fraude), replace=False)])
idx_over = np.concatenate([idx_normal, rng.choice(idx_fraude, size=len(idx_normal), replace=True)])

def novo_lr():
    return Pipeline([("scaler", StandardScaler()), ("modelo", LogisticRegression(max_iter=1000, random_state=SEED))])

def novo_xgb(**extra):
    params = dict(n_estimators=400, max_depth=5, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                  eval_metric="aucpr", tree_method="hist", n_jobs=-1, random_state=SEED)
    params.update(extra)
    return XGBClassifier(**params)

amostragem = []
for estrategia, idx in [("sem amostragem", np.arange(len(y_train))), ("undersampling", idx_under), ("oversampling", idx_over)]:
    for nome_modelo, fabrica in [("LR", novo_lr), ("XGBoost", novo_xgb)]:
        m = fabrica().fit(X_train.iloc[idx], y_train[idx])
        linha = avaliar(f"{nome_modelo} | {estrategia}", y_test, m.predict_proba(X_test)[:, 1])
        linha["linhas de treino"] = len(idx)
        amostragem.append(linha)
tabela_amostragem = pd.DataFrame(amostragem)
tabela_amostragem

### 6.2 Random Forest no Pipeline com ajuste de limiar (e o mesmo para LR e XGBoost)

Escolher o limiar olhando o conjunto de teste é vazamento: o número final ficaria otimista. Aqui o limiar é escolhido com **previsões fora da dobra** (`cross_val_predict`, 3 dobras) dentro do treino, e só depois aplicado ao teste.

O critério é o **F2**, que dá ao recall o dobro do peso da precisão. Ele segue a prioridade do desafio (pegar fraude), mas não deixa a precisão ir a zero como o recall puro.

In [ ]:
cv3 = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)

def escolher_limiar(y_true, proba, beta=2):
    prec, rec, limiares = precision_recall_curve(y_true, proba)
    fbeta = (1 + beta**2) * prec * rec / np.maximum(beta**2 * prec + rec, 1e-12)
    i = int(np.nanargmax(fbeta[:-1]))
    return float(limiares[i])

candidatos = {
    "LR Pipeline": novo_lr(),
    "RF Pipeline (balanced)": Pipeline([("scaler", StandardScaler()),
                                         ("modelo", RandomForestClassifier(n_estimators=200, class_weight="balanced",
                                                                           min_samples_leaf=2, n_jobs=-1, random_state=SEED))]),
    "XGBoost": novo_xgb(),
}

limiares_oof, finais, probas_finais = {}, [], {}
for nome, modelo in candidatos.items():
    inicio = perf_counter()
    oof = cross_val_predict(modelo, X_train, y_train, cv=cv3, method="predict_proba", n_jobs=1)[:, 1]
    limiares_oof[nome] = escolher_limiar(y_train, oof)
    modelo.fit(X_train, y_train)
    probas_finais[nome] = modelo.predict_proba(X_test)[:, 1]
    for t in [0.5, 0.3, limiares_oof[nome]]:
        finais.append(avaliar(nome, y_test, probas_finais[nome], round(t, 3)))
    print(f"{nome:<24} limiar F2 (fora da dobra) = {limiares_oof[nome]:.3f}  [{perf_counter() - inicio:.0f}s]")

tabela_limiar = pd.DataFrame(finais)
tabela_limiar

### 6.3 Busca de hiperparâmetros maior para o XGBoost

Uma grade pequena (profundidade × taxa de aprendizado × peso da fraude) com validação cruzada de 3 dobras, escolhendo pela **PR-AUC**, que mede a qualidade do ranking em todos os limiares. O limiar é decidido depois, pelo F2.

In [ ]:
grade_xgb = {
    "max_depth": [3, 5, 7],
    "learning_rate": [0.05, 0.1],
    "scale_pos_weight": [1, 5, 25],
}
grid_xgb = GridSearchCV(novo_xgb(n_estimators=300, n_jobs=1), grade_xgb, cv=cv3, n_jobs=-1, refit="PR-AUC",
                        scoring={"recall": "recall", "precisão": "precision", "F1": "f1", "PR-AUC": "average_precision"})
inicio = perf_counter()
grid_xgb.fit(X_train, y_train)
print(f"GridSearchCV XGBoost em {perf_counter() - inicio:.0f}s | melhor por PR-AUC: {grid_xgb.best_params_}")

r = pd.DataFrame(grid_xgb.cv_results_)
pd.DataFrame({
    "max_depth": r["param_max_depth"], "learning_rate": r["param_learning_rate"],
    "scale_pos_weight": r["param_scale_pos_weight"],
    "recall": r["mean_test_recall"], "precisão": r["mean_test_precisão"],
    "F1": r["mean_test_F1"], "PR-AUC": r["mean_test_PR-AUC"],
}).sort_values("PR-AUC", ascending=False).head(8)

### 6.4 Variáveis de comportamento ao longo do tempo

O mesmo XGBoost, agora com hora relativa, segundos desde a transação anterior, transações no último minuto e valor comparado com as 10 transações anteriores.

In [ ]:
FEATURES_T = FEATURES + TIME_FEATURES
xgb_tempo = novo_xgb(**grid_xgb.best_params_)
oof_t = cross_val_predict(xgb_tempo, train_df[FEATURES_T], y_train, cv=cv3, method="predict_proba", n_jobs=1)[:, 1]
limiar_t = escolher_limiar(y_train, oof_t)
xgb_tempo.fit(train_df[FEATURES_T], y_train)
proba_tempo = xgb_tempo.predict_proba(test_df[FEATURES_T])[:, 1]

xgb_sem = novo_xgb(**grid_xgb.best_params_)
oof_s = cross_val_predict(xgb_sem, X_train, y_train, cv=cv3, method="predict_proba", n_jobs=1)[:, 1]
limiar_s = escolher_limiar(y_train, oof_s)
xgb_sem.fit(X_train, y_train)
proba_sem = xgb_sem.predict_proba(X_test)[:, 1]

pd.DataFrame([
    avaliar("XGBoost ajustado, sem tempo", y_test, proba_sem, round(limiar_s, 3)),
    avaliar("XGBoost ajustado, com tempo", y_test, proba_tempo, round(limiar_t, 3)),
])

### 6.5 Outros modelos de classificação

`HistGradientBoostingClassifier` (boosting do próprio scikit-learn) e `ExtraTreesClassifier`, com o mesmo procedimento de limiar fora da dobra.

In [ ]:
outros = {
    "HistGradientBoosting": HistGradientBoostingClassifier(max_iter=300, learning_rate=0.1, random_state=SEED),
    "Extra Trees (balanced)": ExtraTreesClassifier(n_estimators=200, class_weight="balanced", min_samples_leaf=2,
                                                   n_jobs=-1, random_state=SEED),
}
for nome, modelo in outros.items():
    inicio = perf_counter()
    oof = cross_val_predict(modelo, X_train, y_train, cv=cv3, method="predict_proba", n_jobs=1)[:, 1]
    limiares_oof[nome] = escolher_limiar(y_train, oof)
    modelo.fit(X_train, y_train)
    probas_finais[nome] = modelo.predict_proba(X_test)[:, 1]
    print(f"{nome:<24} limiar F2 = {limiares_oof[nome]:.3f}  [{perf_counter() - inicio:.0f}s]")

limiares_oof["XGBoost ajustado"] = limiar_s
probas_finais["XGBoost ajustado"] = proba_sem
limiares_oof["XGBoost ajustado + tempo"] = limiar_t
probas_finais["XGBoost ajustado + tempo"] = proba_tempo

### Comparação final

Todos os modelos no conjunto de teste, cada um no limiar escolhido pelo F2 fora da dobra, ordenados por recall.

In [ ]:
comparacao = pd.DataFrame([avaliar(n, y_test, p, round(limiares_oof[n], 3)) for n, p in probas_finais.items()])
comparacao = comparacao.sort_values(["recall", "precisão"], ascending=False).reset_index(drop=True)
comparacao

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
for nome, proba in probas_finais.items():
    prec, rec, _ = precision_recall_curve(y_test, proba)
    ax.plot(rec, prec, label=f"{nome} (AP={average_precision_score(y_test, proba):.3f})")
    pred = proba >= limiares_oof[nome]
    ax.scatter(recall_score(y_test, pred), precision_score(y_test, pred), s=40)
ax.set(title="Curvas de precisão e recall (pontos = limiar escolhido)", xlabel="recall", ylabel="precisão",
       xlim=(0.5, 1.0), ylim=(0, 1.02))
ax.legend(fontsize=8, loc="lower left")
plt.tight_layout()
plt.show()

## 7. Explicar

### 7.1 Importância das variáveis

A importância nativa das árvores diz quais variáveis o modelo mais usou, mas não diz em que direção elas empurram a decisão.

In [ ]:
rf_pipe = candidatos["RF Pipeline (balanced)"]
imp_rf = pd.Series(rf_pipe.named_steps["modelo"].feature_importances_, index=FEATURES).sort_values(ascending=False)
imp_xgb = pd.Series(xgb_sem.feature_importances_, index=FEATURES).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
imp_rf.head(15)[::-1].plot.barh(ax=axes[0], color="#4C72B0", title="Random Forest: importância (impureza)")
imp_xgb.head(15)[::-1].plot.barh(ax=axes[1], color="#DD8452", title="XGBoost ajustado: importância (ganho)")
plt.tight_layout()
plt.show()

### 7.2 SHAP

O SHAP decompõe a pontuação de cada transação na contribuição de cada variável. Uso o `TreeExplainer` no XGBoost ajustado (sem as variáveis de tempo), numa amostra do teste com **todas as fraudes** e 3.000 transações normais. Os valores estão em log-odds: positivo empurra para fraude, negativo para normal.

In [ ]:
amostra_idx = np.concatenate([np.where(y_test == 1)[0],
                              rng.choice(np.where(y_test == 0)[0], size=3000, replace=False)])
X_amostra = X_test.iloc[amostra_idx]
explainer = shap.TreeExplainer(xgb_sem)
shap_values = explainer(X_amostra)

shap.plots.beeswarm(shap_values, max_display=15, show=False)
plt.title("SHAP: impacto de cada variável na pontuação de fraude")
plt.tight_layout()
plt.show()

In [ ]:
shap.plots.bar(shap_values, max_display=15, show=False)
plt.title("SHAP: impacto médio absoluto")
plt.tight_layout()
plt.show()

In [ ]:
proba_amostra = xgb_sem.predict_proba(X_amostra)[:, 1]
y_amostra = y_test[amostra_idx]
pred_amostra = proba_amostra >= limiar_s

i_fraude = int(np.argmax(np.where(y_amostra == 1, proba_amostra, -1)))
print(f"Fraude real com maior pontuação: probabilidade {proba_amostra[i_fraude]:.4f}")
shap.plots.waterfall(shap_values[i_fraude], max_display=12, show=False)
plt.title("Por que esta transação foi marcada como fraude")
plt.tight_layout()
plt.show()

In [ ]:
falsos_pos = np.where((y_amostra == 0) & pred_amostra)[0]
fraudes_perdidas = np.where((y_amostra == 1) & ~pred_amostra)[0]
print(f"Na amostra: {len(falsos_pos)} falsos positivos e {len(fraudes_perdidas)} fraudes perdidas no limiar {limiar_s:.3f}")

if len(fraudes_perdidas):
    j = int(fraudes_perdidas[np.argmin(proba_amostra[fraudes_perdidas])])
    print(f"Fraude perdida com menor pontuação: probabilidade {proba_amostra[j]:.4f}")
    shap.plots.waterfall(shap_values[j], max_display=12, show=False)
    plt.title("Uma fraude que o modelo deixou passar")
    plt.tight_layout()
    plt.show()

In [ ]:
top4 = pd.Series(np.abs(shap_values.values).mean(0), index=FEATURES).sort_values(ascending=False).head(4).index
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, var in zip(axes, top4):
    shap.plots.scatter(shap_values[:, var], ax=ax, show=False, color=shap_values[:, var])
    ax.set_title(var)
plt.tight_layout()
plt.show()

## Conclusões

O resumo dos resultados, do limiar escolhido e do que o SHAP mostrou está no `README.md` do repositório.